# W11 Lab — The Korean Law MCP Server

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/stml_2026/blob/main/lectures/week11/W11_lab_mcp.ipynb)

**Goal.** Do what the Week 11 practice did with the KOSIS server on a second server, Korean Law MCP: connect, get the tools, and give them to an agent that answers a question about a statute and about a court decision. At the end the agent gets two tools of your own next to the tools of the server.

| Part | What you do | Main names |
|---|---|---|
| 1. The Korean law server | get the tools of the server; an agent for a statute and for a court decision | `MCPAdapter`, `list_tools`, `create_agent` |
| 2. Our own tools with the server's tools | one agent with the tools of the server, a calculator, and a tool that saves a file | `@tool` |

**Cell labels.** The first line of each code cell says what to do with it:

- `# ▶ RUN · example`: read each line, then run.
- `# ▶ RUN · setup`: just run.
- `# ✍️ TYPE`: write the code between the `FILL IN` lines. The comment in the block and the *Do:* line before it say what to write.
- `# ▶ RUN · check`: just run; it records whether your code works.

**Submission.** This lab is collected and graded. It has four blocks for you to write: your name and student ID (Part 0), and 1.1, 1.2, and 2.2. Run the notebook from top to bottom. The last cell prints PASS or FAIL for each check, and LAB COMPLETE when all of them pass. Then download the notebook with its outputs (**File → Download → Download .ipynb**) and upload it to the LMS. Due date: see the LMS.

It takes about 40 minutes. Do the Week 11 practice first.

The server is public and needs no key. Without a key, all users of the public copy share one quota. If the server refuses your calls, get a free key at [open.law.go.kr](https://open.law.go.kr) and add `?oc=` and your key to the end of `LAW_URL`.

## MCP in LangChain

A tool made with `@tool` is a Python function inside one program. Another program cannot use it, so every program that needs the same data writes its own tool. The **Model Context Protocol (MCP)** is an open standard for the connection between an AI application and external systems. With MCP, a tool is written once, in a server, and every application that uses the protocol can call it.

MCP has three participants.

| Participant | Definition | In this notebook |
|---|---|---|
| MCP server | A program that provides context to MCP clients. | a public service on the internet |
| MCP client | A component that maintains a connection to one MCP server and obtains context from it for the host. | `MCPAdapter` |
| MCP host | The AI application that coordinates and manages one or more MCP clients. | this notebook, with its agent |

Tools are one kind of context that a server provides, and this notebook uses only tools.

The use of a server has three steps, and they are the same for every server.

| Step | Code |
|---|---|
| Connect to the server | `async with MCPAdapter(url) as adapter:` |
| Ask for its tools | `tools = await adapter.list_tools()` |
| Give the tools to an agent | `agent = create_agent(model, tools)` |

The tools in the list are ordinary LangChain tools. When the model calls one, the call goes to the server, and the reply of the server becomes the tool message.

A call to a server goes over the network, so these functions are asynchronous. `await` before a call waits for its reply. A notebook cell can use `await` directly.

Definitions: MCP documentation, *Architecture overview*.

## 0. Setup

*Do:* run the four cells. Paste your API key into the second cell, and write your name and student ID into the fourth. The third cell makes the model object.

In [ ]:
# ▶ RUN · setup: just run
%pip install -q "langchain[mcp]==1.4.4" "langchain-openai==1.7.0" "langgraph==1.2.14"

In [ ]:
# ▶ RUN · setup: just run
import os

OPENAI_API_KEY = "PASTE-YOUR-KEY-HERE"   # paste your key between the quotes
if OPENAI_API_KEY != "PASTE-YOUR-KEY-HERE":
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY   # LangChain reads the key from here

In [ ]:
# ▶ RUN · setup: just run
from langchain.chat_models import init_chat_model

MODEL = "openai:gpt-4.1"
model = init_chat_model(MODEL)                     # one model object for the whole notebook

In [ ]:
# ✍️ TYPE · exercise: write the code between the FILL IN lines yourself
### FILL IN (START) ###
STUDENT_NAME = ""   # your name
STUDENT_ID = ""     # your student ID
### FILL IN (END) ###
done = {"1.1": False, "1.2": False, "2.2": False}   # the check cells record your results here
print("submitting as:", STUDENT_NAME, STUDENT_ID)

## 1. The Korean Law Server

The Ministry of Government Legislation (법제처) serves Korea's laws, administrative rules, and court decisions through an Open API. Korean Law MCP is an open-source MCP server over that API. A public official built it, and a public copy of it runs at the address below. The steps are those of the practice; only the server is new.

### 1.1 ✍️ The tools of the server

The server has ten tools. Four of them are used below: `search_law` finds a statute by its name and returns its identifier, `get_law_text` reads the articles of a statute, `search_decisions` finds court decisions, and `get_decision_text` reads one decision.

*Do:* replace the line in the block with the call that asks the server for its tools: `adapter.list_tools()`, with `await` before it. Then run the cell. It prints the name of each tool and the names of its arguments.

In [ ]:
# ✍️ TYPE · exercise: write the code between the FILL IN lines yourself
from langchain.mcp import MCPAdapter               # this import prints one warning: this part of LangChain is in beta

LAW_URL = "https://mcp.gomdori.app/law"            # with your own key: "https://mcp.gomdori.app/law?oc=YOUR-KEY"

async with MCPAdapter(LAW_URL) as adapter:         # connect; the connection closes at the end of the block
    ### FILL IN (START) ###
    law_tools = []                                 # replace [] with the call that asks the server for its tools
    ### FILL IN (END) ###

for tool in law_tools:
    print(tool.name, list(tool.args))              # the name of the tool, then the names of its arguments

In [ ]:
# ▶ RUN · check: just run
names = []
for tool in law_tools:
    names.append(tool.name)
done["1.1"] = "search_law" in names                # the list holds the tools of the server
print(done["1.1"])

### 1.2 ✍️ A question about a statute

`show` prints the path of a run in short form, as in the practice: each tool call, the length of each tool result, and the answer.

*Do:* replace the line in the block, so that `create_agent` gets the model, `law_tools`, and `system_prompt=LAW_SYSTEM`. Then run the cell. Read the path: the agent first finds the statute, then reads one article of it.

In [ ]:
# ✍️ TYPE · exercise: write the code between the FILL IN lines yourself
import datetime
from langchain.agents import create_agent
from langchain.messages import HumanMessage

def show(result):
    """Print the path of one run: the tool calls, the length of each tool result, and the answer."""
    for message in result["messages"]:
        if message.type == "ai":
            for call in message.tool_calls:        # one AI message can ask for several tools
                print("TOOL CALL  :", call["name"], call["args"])
        if message.type == "tool":
            print("TOOL RESULT:", message.name, "-", len(message.text), "characters")
    print()
    print(result["messages"][-1].text)             # the last message is the answer

LAW_SYSTEM = """You answer questions about Korean law.
Use the tools, and name the article or the case number that each fact comes from."""
LAW_SYSTEM = LAW_SYSTEM + " Today is " + str(datetime.date.today()) + "."   # the model does not know today's date

### FILL IN (START) ###
law_agent = create_agent(model, [], system_prompt=LAW_SYSTEM)   # replace [] so that the agent gets the tools of the server
### FILL IN (END) ###

STATUTE_QUESTION = "근로기준법에서 연차 유급휴가는 며칠인가?"
result = await law_agent.ainvoke({"messages": [HumanMessage(STATUTE_QUESTION)]})   # the tools are asynchronous, so the agent runs with ainvoke
show(result)

In [ ]:
# ▶ RUN · check: just run
done["1.2"] = "tools" in law_agent.get_graph().nodes   # an agent with tools has a node that runs them
print(done["1.2"])

### 1.3 A question about a court decision

*Do:* run the cell. The same agent gets the case number of a Supreme Court decision, and the model calls other tools of the same server.

In [ ]:
# ▶ RUN · example: read each line, then run
CASE_QUESTION = "대법원 2023다302838 판결의 요지를 알려줘."
result = await law_agent.ainvoke({"messages": [HumanMessage(CASE_QUESTION)]})
show(result)

## 2. Our Own Tools with the Server's Tools

The law server returns the text of statutes and court decisions. It does not calculate, and it cannot write a file in this notebook. A tool from a server and a tool made with `@tool` are the same kind of object, so one agent can have both.

### 2.1 Two tools of our own

*Do:* run the cell. `calculate` is the calculator tool of Week 10. `save_memo` writes a text file into the folder of the notebook.

In [ ]:
# ▶ RUN · example: read each line, then run
from langchain.tools import tool

@tool
def calculate(expression: str) -> str:
    """Evaluate an arithmetic expression and return the result.

    Args:
        expression: A Python arithmetic expression, for example "58 * 94 + 37 * 21".
    """
    try:
        return str(eval(expression, {"__builtins__": {}}))   # run the text as Python, without built-in functions
                                                             # (enough for this lab, not a real safeguard)
    except Exception as error:
        return "error: " + str(error)

@tool
def save_memo(filename: str, text: str) -> str:
    """Save a memo as a text file.

    Args:
        filename: Name of the file, for example "memo.md".
        text: The whole text of the memo.
    """
    file = open(filename, "w", encoding="utf-8")   # "w": make the file, or replace an old one
    file.write(text)
    file.close()
    return "saved " + filename

print(calculate.invoke({"expression": "58 * 94 + 37 * 21"}))

### 2.2 ✍️ One agent with both kinds of tools

*Do:* replace the line in the block, so that `agent_tools` is one list with the tools of the law server, `calculate`, and `save_memo`. Then read `PAY_SYSTEM` and `PAY_QUESTION`, and run the cell. Read the path: which steps use the tools of the server, and which use our own tools. The last lines print the file that the agent saved.

In [ ]:
# ✍️ TYPE · exercise: write the code between the FILL IN lines yourself
### FILL IN (START) ###
agent_tools = law_tools                            # replace this line: the tools of the server and our two tools in one list
### FILL IN (END) ###

PAY_SYSTEM = LAW_SYSTEM + " Do every calculation with the calculate tool."   # one more rule, for our own tool
pay_agent = create_agent(model, agent_tools, system_prompt=PAY_SYSTEM)

PAY_QUESTION = "통상시급이 11,870원인 근로자가 이번 달에 연장근로를 13.5시간 했다. 근로기준법에 따른 연장근로수당을 계산하고, 근거 조문과 계산식을 overtime.md 파일로 저장해줘."
result = await pay_agent.ainvoke({"messages": [HumanMessage(PAY_QUESTION)]})
show(result)

if os.path.exists("overtime.md"):                  # the file that save_memo wrote
    print("----- overtime.md -----")
    print(open("overtime.md", encoding="utf-8").read())

In [ ]:
# ▶ RUN · check: just run
names = []
for tool in agent_tools:
    names.append(tool.name)
done["2.2"] = "search_law" in names and "calculate" in names and "save_memo" in names   # both kinds of tools are in the list
print(done["2.2"])

## Completion

*Do:* run the last cell after all the others. Each line is PASS or FAIL; LAB COMPLETE means that all checks passed. Then download the notebook with its outputs and upload it to the LMS.

In [ ]:
# ▶ RUN · check: just run
checks = {
    "name and student ID filled in (0)":            STUDENT_NAME != "" and STUDENT_ID != "",
    "the tools of the server (1.1)":                done["1.1"],
    "the agent with the tools (1.2)":               done["1.2"],
    "the server's tools and our own tools (2.2)":   done["2.2"],
}
passed = 0
for name in checks:
    if checks[name]:
        print("PASS", name)
        passed = passed + 1
    else:
        print("FAIL", name)
print("submitted by:", STUDENT_NAME, STUDENT_ID)
if passed == len(checks):
    print("LAB COMPLETE")
else:
    print("NOT COMPLETE YET")